# Transformer — 학습 및 구현 테스트 기록

이 노트북은 Transformer를 **단계별로 직접 구현하고 테스트하면서 남긴 원본 학습 기록**을 기반으로 합니다. 각각의 섹션은 당시 테스트를 독립적으로 실행하기 위해 모델 코드를 반복해서 포함할 수 있습니다. 그 과정에서 작성한 코드·주석·실행 결과를 그대로 남겼습니다.

- [최종 Python 구현](https://github.com/Lymonada/research-prep/blob/main/pytorch_basics/12_Transformer.py)
- [Transformer 최종 학습 노트](https://github.com/Lymonada/research-prep/blob/main/notes/10_Transformer_Implementation_Final_Notes.md)
- [GitHub 저장소](https://github.com/Lymonada/research-prep)

> 원본 Colab 실행 결과는 과거에 GPU 환경에서 실행하며 저장한 출력으로, 이번 공개용 파일을 만들면서 재학습하거나 새 코드를 추가하지 않았습니다.


## Token Embedding + Positional Encoding Test

In [ ]:
import math

import torch
import torch.nn as nn


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x


if __name__ == "__main__":
    # Dummy settings
    vocab_size = 20
    d_model = 8
    max_len = 10

    # Batch = 2, Sequence Length = 5
    x = torch.tensor([
        [3, 8, 15, 2, 1],
        [4, 7, 3, 9, 1]
    ])

    # Modules
    token_embedding = TokenEmbedding(
        vocab_size=vocab_size,
        d_model=d_model
    )

    positional_encoding = PositionalEncoding(
        d_model=d_model,
        max_len=max_len
    )

    # 1. Token IDs
    print("Token IDs shape:")
    print(x.shape)

    # 2. Embedding
    embedded = token_embedding(x)

    print("\nEmbedding shape:")
    print(embedded.shape)

    # 3. Positional Encoding
    encoded = positional_encoding(embedded)

    print("\nAfter Positional Encoding:")
    print(encoded.shape)

    # Additional Test
    print("\nEmbedding first token:")
    print(embedded[0, 0])

    print("\nAfter PE first token:")
    print(encoded[0, 0])

    # Shape check
    assert x.shape == (2, 5)
    assert embedded.shape == (2, 5, 8)
    assert encoded.shape == (2, 5, 8)

    print("\nAll shape checks passed!")

Token IDs shape:
torch.Size([2, 5])

Embedding shape:
torch.Size([2, 5, 8])

After Positional Encoding:
torch.Size([2, 5, 8])

Embedding first token:
tensor([-4.6213, -1.4353,  0.6863,  2.0587,  2.2642, -2.3469,  4.0327, -3.6227],
       grad_fn=<SelectBackward0>)

After PE first token:
tensor([-4.6213, -0.4353,  0.6863,  3.0587,  2.2642, -1.3469,  4.0327, -2.6227],
       grad_fn=<SelectBackward0>)

All shape checks passed!


## Attention() Test


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V):
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, 2, 3)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights


####################
# attention() Test #
#####################

if __name__ == "__main__":
    B=2
    h=4
    T_q=5
    T_k=7
    d_k=8
    d_v=6
    Q = torch.randn(B, h, T_q, d_k)
    K = torch.randn(B, h, T_k, d_k)
    V = torch.randn(B, h, T_k, d_v)

    output , att_weights = attention(Q, K, V)
    print(output.shape)
    print(att_weights.shape)
    print(att_weights.sum(dim=-1))

torch.Size([2, 4, 5, 6])
torch.Size([2, 4, 5, 7])
tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]])


## Multihead Attention Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V):
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_k], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value):
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head)) # 마지막 차원을 num_heads, d_head로 쪼개기
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B,h,T_q,d_head] -> [B, T, d_model]
        output, att_weights = attention(Q,K,V)
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) #

        output = self.W_O(output)

        return output, att_weights

##############################
# MultiHeadAttention Test
##############################

if __name__ == "__main__":
    B = 2
    T_q = 5
    T_k = 7
    d_model = 32
    num_heads = 4

    query = torch.randn(B, T_q, d_model)
    key = torch.randn(B, T_k, d_model)
    value = torch.randn(B, T_k, d_model)

    mha = MultiHeadAttention(d_model, num_heads)

    output, att_weights = mha(query, key, value)

    print(output.shape)
    print(att_weights.shape)
    print(att_weights.sum(dim=-1))

torch.Size([2, 5, 32])
torch.Size([2, 4, 5, 7])
tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)


## Encoder Block Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V): # Expected Q/K/V shape: [B, h, T, d_head]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_k], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B,h,T_q,d_head] -> [B, T, d_model]
        output, att_weights = attention(Q,K,V)
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) #

        output = self.W_O(output)

        return output, att_weights

# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output

# 7. Encoder Block
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x):
        attn_output, attn_weights = self.mha(x,x,x) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights



#####################
# Encoder Block test #
#####################

B = 2
S = 5          # source sequence length
d_model = 8
num_heads = 2
d_ff = 32

x = torch.randn(B, S, d_model)

encoder_block = EncoderBlock(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff
)

output, attn_weights = encoder_block(x)

print("input shape       :", x.shape)
print("output shape      :", output.shape)
print("attention shape   :", attn_weights.shape)

# 각 query가 key 방향으로 만든 attention weight의 합
print("attention sum     :", attn_weights.sum(dim=-1))


input shape       : torch.Size([2, 5, 8])
output shape      : torch.Size([2, 5, 8])
attention shape   : torch.Size([2, 2, 5, 5])
attention sum     : tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)


## Encoder Test


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V): # Expected Q/K/V shape: [B, h, T, d_head]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_k], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B,h,T_q,d_head] -> [B, T, d_model]
        output, att_weights = attention(Q,K,V)
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) #

        output = self.W_O(output)

        return output, att_weights

# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output

# 7. Encoder layer
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x):
        attn_output, attn_weights = self.mha(x,x,x) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x):
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights


################
# Encoder test #
################

B = 2
S = 5          # source sequence length
d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3
0
# [B, T, d_model]
x = torch.randn(B, S, d_model)

encoder = Encoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
)

output, all_attn_weights = encoder(x)

print("input shape :", x.shape)
print("output shape:", output.shape)

print("number of attention matrices:", len(all_attn_weights))

for i, attn_weights in enumerate(all_attn_weights):
    print(f"layer {i+1} attention shape:", attn_weights.shape)
    print(f"layer {i+1} attention sum:")
    print(attn_weights.sum(dim=-1))


input shape : torch.Size([2, 5, 8])
output shape: torch.Size([2, 5, 8])
number of attention matrices: 3
layer 1 attention shape: torch.Size([2, 2, 5, 5])
layer 1 attention sum:
tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)
layer 2 attention shape: torch.Size([2, 2, 5, 5])
layer 2 attention sum:
tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)
layer 3 attention shape: torch.Size([2, 2, 5, 5])
layer 3 attention sum:
tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_

## Decoder Block Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V): # Expected Q/K/V shape: [B, h, T, d_head]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_k], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B,h,T_q,d_head] -> [B, T, d_model]
        output, att_weights = attention(Q,K,V)
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) #

        output = self.W_O(output)

        return output, att_weights

# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output

# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)
        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)
        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B, h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights




#####################
# DecoderBlock test #
#####################

B = 2
T = 5          # target length
S = 7          # source length
d_model = 8
num_heads = 2
d_ff = 32

decoder_block = DecoderBlock(d_model, num_heads, d_ff)

x = torch.randn(B, T, d_model)
encoder_output = torch.randn(B, S, d_model)

output, self_attn_weights, cross_attn_weights = decoder_block(
    x,
    encoder_output
)

print("output:", output.shape)
print("self attention:", self_attn_weights.shape)
print("cross attention:", cross_attn_weights.shape)

print("self attn sum:", self_attn_weights.sum(dim=-1))
print("cross attn sum:", cross_attn_weights.sum(dim=-1))


output: torch.Size([2, 5, 8])
self attention: torch.Size([2, 2, 5, 5])
cross attention: torch.Size([2, 2, 5, 7])
self attn sum: tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)
cross attn sum: tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)


## Decoder Test


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V): # Expected Q/K/V shape: [B, h, T, d_head]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_k], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B,h,T_q,d_head] -> [B, T, d_model]
        output, att_weights = attention(Q,K,V)
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) #

        output = self.W_O(output)

        return output, att_weights

# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output

# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)
        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)
        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B, h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights




################
# Decoder test #
################

B = 2
T = 5          # target sequence length
S = 7          # source sequence length
d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3

x = torch.randn(B, T, d_model)
encoder_output = torch.randn(B, S, d_model)

decoder = Decoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
)

output, all_self_attn_weights, all_cross_attn_weights = decoder(
    x,
    encoder_output
)

print("input shape           :", x.shape)
print("encoder output shape  :", encoder_output.shape)
print("decoder output shape  :", output.shape)

print("num decoder layers    :", len(all_self_attn_weights))

for i in range(num_layers):
    print(f"\nLayer {i+1}")
    print("self attention shape  :", all_self_attn_weights[i].shape)
    print("cross attention shape :", all_cross_attn_weights[i].shape)

    print(
        "self attn sum         :",
        all_self_attn_weights[i].sum(dim=-1)
    )

    print(
        "cross attn sum        :",
        all_cross_attn_weights[i].sum(dim=-1)
    )


input shape           : torch.Size([2, 5, 8])
encoder output shape  : torch.Size([2, 7, 8])
decoder output shape  : torch.Size([2, 5, 8])
num decoder layers    : 3

Layer 1
self attention shape  : torch.Size([2, 2, 5, 5])
cross attention shape : torch.Size([2, 2, 5, 7])
self attn sum         : tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)
cross attn sum        : tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)

Layer 2
self attention shape  : torch.Size([2, 2, 5, 5])
cross attention shape : torch.Size([2, 2, 5, 7])
self attn sum         : tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1

## Causal Mask Test


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V, mask=None): # Expected shapes: Q/K/V : [B, h, T, d_head], mask: [1, 1, T_q, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 masked attention이라면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    if (mask != None): att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 11. Masks
def create_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    return mask

B = 1
h = 1
T = 5
d_k = 4

Q = torch.randn(B, h, T, d_k)
K = torch.randn(B, h, T, d_k)
V = torch.randn(B, h, T, d_k)

mask = create_causal_mask(T)

output, att_weights = attention(Q, K, V, mask)

print(att_weights[0, 0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4284, 0.5716, 0.0000, 0.0000, 0.0000],
        [0.6002, 0.1396, 0.2603, 0.0000, 0.0000],
        [0.1155, 0.3490, 0.2893, 0.2462, 0.0000],
        [0.0688, 0.1492, 0.5078, 0.2310, 0.0432]])


## Padding Mask Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V, mask=None): # Expected shapes: Q/K/V : [B, h, T, d_head], mask: [1, 1, T_q, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if (mask != None): att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.
    # 텐서 하나와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx # pad_idx가 아닌 원소는 True, pad_idx랑 같으면 False를 가지는 텐서를 만들어줌 = mask
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    return mask # 이렇게 함으로써 각 배치에서 모든 head에서 모든 query가 특정 key를 못보게 만듦.

#####################
# Padding Mask Test #
#####################

B = 2
h = 1
T = 5
d_k = 4

Q = torch.randn(B, h, T, d_k)
K = torch.randn(B, h, T, d_k)
V = torch.randn(B, h, T, d_k)

seq = torch.tensor([
    [5, 8, 2, 9, 4],
    [7, 3, 6, 0, 0]
])

padding_mask = create_padding_mask(seq, 0)
output, att_weights = attention(
    Q,
    K,
    V,
    mask=padding_mask
)
print(att_weights[1, 0])
print(att_weights[0, 0])


tensor([[0.6751, 0.2801, 0.0448, 0.0000, 0.0000],
        [0.3307, 0.1904, 0.4789, 0.0000, 0.0000],
        [0.1150, 0.0400, 0.8450, 0.0000, 0.0000],
        [0.4587, 0.5168, 0.0245, 0.0000, 0.0000],
        [0.2058, 0.0438, 0.7504, 0.0000, 0.0000]])
tensor([[0.3395, 0.0065, 0.2366, 0.2872, 0.1302],
        [0.0406, 0.8022, 0.0330, 0.0439, 0.0804],
        [0.3781, 0.1306, 0.1489, 0.0861, 0.2562],
        [0.0753, 0.1126, 0.0741, 0.5619, 0.1762],
        [0.1990, 0.2036, 0.3135, 0.1790, 0.1048]])


## Mask Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 11. Masks
def create_causal_mask(seq_len): # masked self attention을 위한 mask. 미래의 토큰을 attend할수 없게 만듦.
    mask = torch.tril(torch.ones(seq_len, seq_len)) # [T, T] shape의 lower-triangular matrix 만듦
    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool() # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T, T] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.
    # 텐서 하나와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.
    return mask # 이렇게 함으로써 각 배치에서 모든 head에서 모든 query가 특정 key를 못보게 만듦.

def create_decoder_mask(target_seq, pad_idx):# 텐서 하나와 PAD token의 정수 id를 받음. target_seq: [B, T]
    seq_len = target_seq.shape[1] # target_seq: [B, T] 에서 T 가져오기
    causal_mask = create_causal_mask(seq_len) # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]
    padding_mask = create_padding_mask(target_seq, pad_idx) # paddin mask: [B, 1, 1, T_k]
    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask

target_seq = torch.tensor([
    [1, 5, 8, 9, 2],  # PAD 없음
    [1, 7, 3, 0, 0]   # 마지막 두 개가 PAD
])

mask = create_decoder_mask(target_seq, pad_idx=0)

print(mask.shape)
print("Sequence 0:")
print(mask[0, 0])

print("\nSequence 1:")
print(mask[1, 0])

torch.Size([2, 1, 5, 5])
Sequence 0:
tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])

Sequence 1:
tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True, False, False]])


## Multihead Attention Mask Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V, mask=None): # Expected shapes: Q/K/V : [B, h, T, d_head], mask: [1, 1, T_q, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B,h,T_q,d_head] -> [B, T, d_model]
        output, att_weights = attention(Q, K, V, mask)
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights

# 11. Masks
def create_causal_mask(seq_len): # masked self attention을 위한 mask. 미래의 토큰을 attend할수 없게 만듦.
    mask = torch.tril(torch.ones(seq_len, seq_len)) # [T, T] shape의 lower-triangular matrix 만듦
    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool() # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T, T] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.
    # 텐서 하나와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.
    return mask # 이렇게 함으로써 각 배치에서 모든 head에서 모든 query가 특정 key를 못보게 만듦.

def create_decoder_mask(target_seq, pad_idx):# 텐서 하나와 PAD token의 정수 id를 받음. target_seq: [B, T]
    seq_len = target_seq.shape[1] # target_seq: [B, T] 에서 T 가져오기
    causal_mask = create_causal_mask(seq_len) # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]
    padding_mask = create_padding_mask(target_seq, pad_idx) # paddin mask: [B, 1, 1, T_k]
    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask

B = 2
T = 5
d_model = 8
num_heads = 2

x = torch.randn(B, T, d_model)

target_seq = torch.tensor([
    [1, 5, 8, 9, 2],
    [1, 7, 3, 0, 0]
])

mask = create_decoder_mask(target_seq, pad_idx=0)
mha = MultiHeadAttention(d_model, num_heads)

output, att_weights = mha(x, x, x, mask=mask)

print(output.shape)
print(att_weights.shape)

print(att_weights[1, 0])

torch.Size([2, 5, 8])
torch.Size([2, 2, 5, 5])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5695, 0.4305, 0.0000, 0.0000, 0.0000],
        [0.2843, 0.3238, 0.3919, 0.0000, 0.0000],
        [0.1868, 0.3416, 0.4716, 0.0000, 0.0000],
        [0.2999, 0.3794, 0.3207, 0.0000, 0.0000]], grad_fn=<SelectBackward0>)


## Encoder Mask Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V, mask=None): # Expected shapes: Q/K/V : [B, h, T, d_head], mask: [1, 1, T_q, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B,h,T_q,d_head] -> [B, T, d_model]
        output, att_weights = attention(Q, K, V, mask)
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights

# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output

# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights

# 11. Masks
def create_causal_mask(seq_len): # masked self attention을 위한 mask. 미래의 토큰을 attend할수 없게 만듦.
    mask = torch.tril(torch.ones(seq_len, seq_len)) # [T, T] shape의 lower-triangular matrix 만듦
    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool() # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T, T] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.
    # 텐서 하나와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.
    return mask # 이렇게 함으로써 각 배치에서 모든 head에서 모든 query가 특정 key를 못보게 만듦.

def create_decoder_mask(target_seq, pad_idx):# 텐서 하나와 PAD token의 정수 id를 받음. target_seq: [B, T]
    seq_len = target_seq.shape[1] # target_seq: [B, T] 에서 T 가져오기
    causal_mask = create_causal_mask(seq_len) # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]
    padding_mask = create_padding_mask(target_seq, pad_idx) # paddin mask: [B, 1, 1, T_k]
    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask

B = 2
S = 5
d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3

x = torch.randn(B, S, d_model)

src_seq = torch.tensor([
    [5, 8, 2, 9, 4],
    [7, 3, 6, 0, 0]
])

src_mask = create_padding_mask(src_seq, pad_idx=0)

encoder = Encoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
)

output, all_attn_weights = encoder(
    x,
    src_mask=src_mask
)

print(output.shape)
print(len(all_attn_weights))

for i, weights in enumerate(all_attn_weights):
    print(f"Layer {i}: {weights.shape}")

print("\nLast layer, sequence 1, head 0:")
print(all_attn_weights[-1][1, 0])



torch.Size([2, 5, 8])
3
Layer 0: torch.Size([2, 2, 5, 5])
Layer 1: torch.Size([2, 2, 5, 5])
Layer 2: torch.Size([2, 2, 5, 5])

Last layer, sequence 1, head 0:
tensor([[0.3063, 0.3145, 0.3792, 0.0000, 0.0000],
        [0.2962, 0.3468, 0.3570, 0.0000, 0.0000],
        [0.3768, 0.3402, 0.2830, 0.0000, 0.0000],
        [0.3580, 0.3667, 0.2753, 0.0000, 0.0000],
        [0.2784, 0.3525, 0.3691, 0.0000, 0.0000]], grad_fn=<SelectBackward0>)


## Decoder Mask Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output


# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights


# 11. Masks
def create_causal_mask(seq_len): # masked self attention을 위한 mask. 미래의 토큰을 attend할수 없게 만듦.
    mask = torch.tril(torch.ones(seq_len, seq_len)) # [T, T] shape의 lower-triangular matrix 만듦
    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool() # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T, T] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.
    # 텐서 하나와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.
    return mask # 이렇게 함으로써 각 배치에서 모든 head에서 모든 query가 특정 key를 못보게 만듦.

def create_decoder_mask(target_seq, pad_idx):# 텐서 하나와 PAD token의 정수 id를 받음. target_seq: [B, T]
    seq_len = target_seq.shape[1] # target_seq: [B, T] 에서 T 가져오기
    causal_mask = create_causal_mask(seq_len) # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]
    padding_mask = create_padding_mask(target_seq, pad_idx) # paddin mask: [B, 1, 1, T_k]
    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask




#####################
# Decoder Mask Test #
#####################

B = 2
T = 5
S = 6
d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3
pad_idx = 0

# decoder input representation
x = torch.randn(B, T, d_model)

# encoder output
encoder_output = torch.randn(B, S, d_model)

# target token ids
target_seq = torch.tensor([
    [1, 5, 8, 9, 2],     # PAD 없음
    [1, 7, 3, 0, 0]      # 마지막 두 개가 PAD
])

# source token ids
src_seq = torch.tensor([
    [4, 6, 8, 2, 9, 3],  # PAD 없음
    [5, 7, 2, 1, 0, 0]   # 마지막 두 개가 PAD
])

# Decoder Self-Attention용:
# causal + target padding
self_mask = create_decoder_mask(target_seq, pad_idx)

# Decoder Cross-Attention용:
# source padding
cross_mask = create_padding_mask(src_seq, pad_idx)

decoder = Decoder(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
)

output, all_self_attn_weights, all_cross_attn_weights = decoder(
    x,
    encoder_output,
    self_mask=self_mask,
    cross_mask=cross_mask
)

print("Decoder output shape:")
print(output.shape)

print("\nNumber of decoder layers:")
print(len(all_self_attn_weights))
print(len(all_cross_attn_weights))

for i in range(num_layers):
    print(f"\nLayer {i}")
    print("Self-attention shape:",
          all_self_attn_weights[i].shape)
    print("Cross-attention shape:",
          all_cross_attn_weights[i].shape)

print("\nLast layer - sequence 1 - head 0 - SELF attention:")
print(all_self_attn_weights[-1][1, 0])

print("\nLast layer - sequence 1 - head 0 - CROSS attention:")
print(all_cross_attn_weights[-1][1, 0])

Decoder output shape:
torch.Size([2, 5, 8])

Number of decoder layers:
3
3

Layer 0
Self-attention shape: torch.Size([2, 2, 5, 5])
Cross-attention shape: torch.Size([2, 2, 5, 6])

Layer 1
Self-attention shape: torch.Size([2, 2, 5, 5])
Cross-attention shape: torch.Size([2, 2, 5, 6])

Layer 2
Self-attention shape: torch.Size([2, 2, 5, 5])
Cross-attention shape: torch.Size([2, 2, 5, 6])

Last layer - sequence 1 - head 0 - SELF attention:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4640, 0.5360, 0.0000, 0.0000, 0.0000],
        [0.2737, 0.3103, 0.4161, 0.0000, 0.0000],
        [0.3685, 0.3736, 0.2578, 0.0000, 0.0000],
        [0.3464, 0.3304, 0.3232, 0.0000, 0.0000]], grad_fn=<SelectBackward0>)

Last layer - sequence 1 - head 0 - CROSS attention:
tensor([[0.2759, 0.2088, 0.3110, 0.2042, 0.0000, 0.0000],
        [0.2587, 0.2274, 0.3279, 0.1860, 0.0000, 0.0000],
        [0.1968, 0.2037, 0.2812, 0.3183, 0.0000, 0.0000],
        [0.3738, 0.2207, 0.2474, 0.1581, 0.0000, 0.0000]

## Transformer Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights





# 11. Masks
def create_causal_mask(seq_len): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기

    causal_mask = create_causal_mask(seq_len)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_len, pad_idx):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)

        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        return (
                decoder_output,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

####################
# Transformer Test #
####################

B = 2
S = 7
T = 5

src_vocab_size = 20
tgt_vocab_size = 30

d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3
max_len = 20
pad_idx = 0

src = torch.tensor([
    [1, 4, 7, 3, 0, 0, 0],
    [2, 5, 8, 9, 3, 4, 0]
])

tgt = torch.tensor([
    [1, 6, 4, 0, 0],
    [1, 3, 7, 8, 0]
])

my_model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_len, pad_idx)

decoder_output,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = my_model(src, tgt)

print("Decoder output shape:")
print(decoder_output.shape)

print(len(encoder_attn_weights))
for i, weights in enumerate(encoder_attn_weights):
    print(f"Layer {i}: {weights.shape}")

print(len(decoder_self_attn_weights))
for i, weights in enumerate(decoder_self_attn_weights):
    print(f"Layer {i}: {weights.shape}")

print(len(decoder_cross_attn_weights))
for i, weights in enumerate(decoder_cross_attn_weights):
    print(f"Layer {i}: {weights.shape}")

Decoder output shape:
torch.Size([2, 5, 8])
3
Layer 0: torch.Size([2, 2, 7, 7])
Layer 1: torch.Size([2, 2, 7, 7])
Layer 2: torch.Size([2, 2, 7, 7])
3
Layer 0: torch.Size([2, 2, 5, 5])
Layer 1: torch.Size([2, 2, 5, 5])
Layer 2: torch.Size([2, 2, 5, 5])
3
Layer 0: torch.Size([2, 2, 5, 7])
Layer 1: torch.Size([2, 2, 5, 7])
Layer 2: torch.Size([2, 2, 5, 7])


## Output Projection and Logits Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights





# 11. Masks
def create_causal_mask(seq_len): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기

    causal_mask = create_causal_mask(seq_len)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_len, pad_idx):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)
        # encoder_output: [B,S,d_model], encoder_attn_weights: [B,h,S,S]
        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        # decoder_output: [B,T,d_model], decoder_self_attn_weights: [B,h,T,T], decoder_cross_attn_weights: [B,h,T,S]
        vocab_logits = self.output_linear(decoder_output)
        # vocab_logits: [B,T,tgt_vocab_size]

        return (
                vocab_logits,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

###########################
# Transformer Logits Test #
###########################

B = 2
S = 7
T = 5

src_vocab_size = 20
tgt_vocab_size = 30

d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3
max_len = 20
pad_idx = 0

src = torch.tensor([
    [1, 4, 7, 3, 0, 0, 0],
    [2, 5, 8, 9, 3, 4, 0]
])

tgt = torch.tensor([
    [1, 6, 4, 0, 0],
    [1, 3, 7, 8, 0]
])

dummy = torch.tensor([
    [4, 7, 9, 0, 0],
    [3, 8, 2, 5, 0]
])

my_model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_len, pad_idx)

vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = my_model(src, tgt)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

logits = vocab_logits.reshape(-1, vocab_logits.size(-1))
flattened_dummy = dummy.reshape(-1)

loss = criterion(logits, flattened_dummy)

print("Logits Shape: ")
print(vocab_logits.shape)

print("Labels Shape:")
print(dummy.shape)

print("Flattened Logits Shape: ")
print(logits.shape)

print("Flattened Labels Shape:")
print(flattened_dummy.shape)

print("Loss: ")
print(loss)

Logits Shape: 
torch.Size([2, 5, 30])
Labels Shape:
torch.Size([2, 5])
Flattened Logits Shape: 
torch.Size([10, 30])
Flattened Labels Shape:
torch.Size([10])
Loss: 
tensor(3.6778, grad_fn=<NllLossBackward0>)


## generate_copy_batch() Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# 13. Copy Task
def generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device): #
    src = torch.randint(3, vocab_size, (batch_size, seq_len), device=device)
    tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
    tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가
    return src, tgt_input, tgt_label

############################
# generate_copy_batch() Test
############################

batch_size = 2
seq_len = 5
vocab_size = 20

bos_idx = 1
eos_idx = 2

device = "cuda" if torch.cuda.is_available() else "cpu"

src, tgt_input, tgt_label = generate_copy_batch(
    batch_size=batch_size,
    seq_len=seq_len,
    vocab_size=vocab_size,
    bos_idx=bos_idx,
    eos_idx=eos_idx,
    device=device
)

print("src:")
print(src)
print("shape:", src.shape)

print("\ntgt_input:")
print(tgt_input)
print("shape:", tgt_input.shape)

print("\ntgt_label:")
print(tgt_label)
print("shape:", tgt_label.shape)

# shape 확인
assert src.shape == (batch_size, seq_len)
assert tgt_input.shape == (batch_size, seq_len + 1)
assert tgt_label.shape == (batch_size, seq_len + 1)

# BOS / EOS 위치 확인
assert torch.all(tgt_input[:, 0] == bos_idx)
assert torch.all(tgt_label[:, -1] == eos_idx)

# 실제 copy 관계 확인
assert torch.equal(tgt_input[:, 1:], src)
assert torch.equal(tgt_label[:, :-1], src)

print("\nAll tests passed!")

src:
tensor([[13, 10, 14, 16, 13],
        [ 8, 13, 13,  5, 14]], device='cuda:0')
shape: torch.Size([2, 5])

tgt_input:
tensor([[ 1, 13, 10, 14, 16, 13],
        [ 1,  8, 13, 13,  5, 14]], device='cuda:0')
shape: torch.Size([2, 6])

tgt_label:
tensor([[13, 10, 14, 16, 13,  2],
        [ 8, 13, 13,  5, 14,  2]], device='cuda:0')
shape: torch.Size([2, 6])

All tests passed!


## Training Loop Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights






# 11. Masks
def create_causal_mask(seq_len, device): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기
    current_device = target_seq.device
    causal_mask = create_causal_mask(seq_len, current_device)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len=5000):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)
        # encoder_output: [B,S,d_model], encoder_attn_weights: list of num_layers, each element: [B,h,S,S]
        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        # decoder_output: [B,T,d_model], decoder_self_attn_weights: list of num_layers, each [B,h,T,T], decoder_cross_attn_weights: list of num_layers, each [B,h,T,S]
        vocab_logits = self.output_linear(decoder_output)
        # vocab_logits: [B,T,tgt_vocab_size]

        return (
                vocab_logits,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

# 13. Copy Task
def generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device): #
    src = torch.randint(3, vocab_size, (batch_size, seq_len), device=device)
    tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
    tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가
    return src, tgt_input, tgt_label # src: [B, S], tgt_input: [B, S+1], tgt_label: [B, S+1]


# 14. Training Loop
vocab_size = 20
pad_idx = 0
bos_idx = 1
eos_idx = 2

seq_len = 5
batch_size = 32

d_model = 32
num_heads = 4
d_ff = 64
num_layers = 2

learning_rate = 1e-3
num_steps = 1000
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = Transformer(vocab_size, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss()

for step in range(num_steps):

    src, tgt_input, tgt_label = generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device)

    optimizer.zero_grad()

    vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(src, tgt_input)


    logits = vocab_logits.reshape(-1, vocab_logits.size(-1))
    label = tgt_label.reshape(-1)

    loss = loss_function(logits, label)

    loss.backward()
    optimizer.step()


    if step % 100 == 0:
        print("Loss: ")
        print(loss)

Loss: 
tensor(3.1634, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.8509, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.1402, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.5385, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.3268, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0994, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0473, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0152, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0200, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0042, device='cuda:0', grad_fn=<NllLossBackward0>)


## Training-Time Prediction Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights






# 11. Masks
def create_causal_mask(seq_len, device): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기
    current_device = target_seq.device
    causal_mask = create_causal_mask(seq_len, current_device)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len=5000):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)
        # encoder_output: [B,S,d_model], encoder_attn_weights: list of num_layers, each element: [B,h,S,S]
        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        # decoder_output: [B,T,d_model], decoder_self_attn_weights: list of num_layers, each [B,h,T,T], decoder_cross_attn_weights: list of num_layers, each [B,h,T,S]
        vocab_logits = self.output_linear(decoder_output)
        # vocab_logits: [B,T,tgt_vocab_size]

        return (
                vocab_logits,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

# 13. Copy Task
def generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device): #
    src = torch.randint(3, vocab_size, (batch_size, seq_len), device=device)
    tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
    tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가
    return src, tgt_input, tgt_label # src: [B, S], tgt_input: [B, S+1], tgt_label: [B, S+1]


# 14. Training Loop
vocab_size = 20
pad_idx = 0
bos_idx = 1
eos_idx = 2

seq_len = 5
batch_size = 32

d_model = 32
num_heads = 4
d_ff = 64
num_layers = 2

learning_rate = 1e-3
num_steps = 1000
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = Transformer(vocab_size, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss()


for step in range(num_steps):

    src, tgt_input, tgt_label = generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device)

    optimizer.zero_grad()

    vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(src, tgt_input)


    logits = vocab_logits.reshape(-1, vocab_logits.size(-1))
    label = tgt_label.reshape(-1)

    loss = loss_function(logits, label)

    loss.backward()
    optimizer.step()


# 15. Training-time prediction test
test_batch_size = 1
model.eval()
with torch.no_grad():
    src, tgt_input, tgt_label = generate_copy_batch(test_batch_size, seq_len, vocab_size, bos_idx, eos_idx, device)
    vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(src, tgt_input)

    prediction = vocab_logits.argmax(dim=-1)

print(src)
print(tgt_input)
print(tgt_label)
print(prediction)
print(torch.equal(prediction, tgt_label))


tensor([[12, 18, 18, 13,  6]], device='cuda:0')
tensor([[ 1, 12, 18, 18, 13,  6]], device='cuda:0')
tensor([[12, 18, 18, 13,  6,  2]], device='cuda:0')
tensor([[12, 18, 18, 13,  6,  2]], device='cuda:0')
True


## Autoregressive Inference(B=1) Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights






# 11. Masks
def create_causal_mask(seq_len, device): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기
    current_device = target_seq.device
    causal_mask = create_causal_mask(seq_len, current_device)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len=5000):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)
        # encoder_output: [B,S,d_model], encoder_attn_weights: list of num_layers, each element: [B,h,S,S]
        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        # decoder_output: [B,T,d_model], decoder_self_attn_weights: list of num_layers, each [B,h,T,T], decoder_cross_attn_weights: list of num_layers, each [B,h,T,S]
        vocab_logits = self.output_linear(decoder_output)
        # vocab_logits: [B,T,tgt_vocab_size]

        return (
                vocab_logits,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

# 13. Copy Task
def generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device): #
    src = torch.randint(3, vocab_size, (batch_size, seq_len), device=device)
    tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
    tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가
    return src, tgt_input, tgt_label # src: [B, S], tgt_input: [B, S+1], tgt_label: [B, S+1]


# 14. Training Loop
vocab_size = 20
pad_idx = 0
bos_idx = 1
eos_idx = 2

seq_len = 5
batch_size = 32

d_model = 32
num_heads = 4
d_ff = 64
num_layers = 2

learning_rate = 1e-3
num_steps = 1000
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = Transformer(vocab_size, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss()


for step in range(num_steps):

    train_src, train_tgt_input, train_tgt_label = generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device)

    optimizer.zero_grad()

    vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(train_src, train_tgt_input)


    logits = vocab_logits.reshape(-1, vocab_logits.size(-1))
    label = train_tgt_label.reshape(-1)

    loss = loss_function(logits, label)

    loss.backward()
    optimizer.step()

#16. Autoregressive Inference
@torch.no_grad()
def greedy_decode(model, src, bos_idx, eos_idx, max_new_tokens, device): # src: [B,S]
    model.eval()
    batch_size = src.shape[0]

    # [B, 1] 크기로 bos_idx 채우기
    generated = torch.full((batch_size, 1), bos_idx, dtype=torch.long, device=device)
    # model에 넣어줄 target sequence로 [BOS]만 있는 텐서 넣기

    # 최대 max_new_tokens개의 새로운 token을 autoregressive하게 생성
    for tokens in range(max_new_tokens):
        vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(src, generated)
        next_token_logits = vocab_logits[:,-1,:] # next_token_logits: [B, vocab_size]
        # vocab_logits에는 모든 timestep의 다음 토큰 예상값이 logits으로 들어있기 때문에
        # 마지막 시점의, 즉 마지막 position의 예상값을 가져오기 위해 T차원을 -1로 가져옴.

        next_token = next_token_logits.argmax(dim=-1) # next_token_logits에서 가장 큰 값의 인덱스는 모델이 예상한 다음 token id. next_token: [B]
        next_token = next_token.unsqueeze(-1) # 제일 안쪽 차원을 하나 추가. next_token: [B, 1]
        generated = torch.cat([generated, next_token], dim=1) # generated와 next_token을 붙임. generated: [B, current_length+1]

        if next_token.item() == eos_idx: # [EOS]를 생성했으면 문장이 끝났기 때문에 break. next_token이 값을 딱 하나만 가지기 때문에 .item()으로 꺼낼 수 있음
            break # 만약 batch_size가 1이 아니라서 next_token에 여러 값이 있다면 작동X

    return generated # generated: [B,T]

# 17. Teacher-forcing vs Autoregressive Inference Test

# 새로운 테스트용 copy sample 1개 생성
test_src, test_tgt_input, test_tgt_label = generate_copy_batch(
    batch_size=1,
    seq_len=seq_len,
    vocab_size=vocab_size,
    bos_idx=bos_idx,
    eos_idx=eos_idx,
    device=device
)

# 1) Teacher-forcing prediction
model.eval()

with torch.no_grad():
    vocab_logits, _, _, _ = model(test_src, test_tgt_input)
    teacher_prediction = vocab_logits.argmax(dim=-1)

# 2) Autoregressive prediction
max_new_tokens = src.shape[1] + 1

generated = greedy_decode(
    model,
    test_src,
    bos_idx,
    eos_idx,
    max_new_tokens,
    device
)

print("src:                 ", test_src)
print("tgt_input:           ", test_tgt_input)
print("tgt_label:           ", test_tgt_label)
print("teacher prediction:  ", teacher_prediction)
print("autoregressive:      ", generated)

# 오케이 이제 transformer를 구현하는 모든 단계를 완료했어.
# 이제 전체 구현에서 핵심 shape과 forward flow를 한 번 최종 정리하고 싶어.
# 그리고 내가 지금까지 너와 대화하면서 한단계씩 transformer를 구현하면서
# 배운 여러가지를 마크다운에 정리도 하고 싶어졌어. 그리고 또 궁금한게,
# 현재까지 내가 구현한 transformer가 표준 transformer와 얼마만큼 다르고
# 어느부분이 다른지도 궁금해. 내가 너무 야매로 구현한건지, 아니면 그래도
# 꽤 정확하게 구현한건지 그런것도 알고 싶네. 이 이후에 전체 구현에서 핵심 shape과 forward flow를  최종 정리를 해보자.

# 실제 transformer랑 비교
# 지금까지 채팅하면서 배운 transformer구현을 마크다운으로 정리 + 어려웠던 부분도
# 현재 transformer에서 test코드 떼어내기
# dsa, computer architecture, c++ 공부 꺼내기
# 현재 copy batch방법은 src에서 tgt_input이랑 tgt_label을 다 생성하는 방식인데, 실제로 transformer에 넣는걸 직접 넣어보는 테스트는? ex)번역


src:                  tensor([[ 9, 17, 16,  9,  8]], device='cuda:0')
tgt_input:            tensor([[ 1,  9, 17, 16,  9,  8]], device='cuda:0')
tgt_label:            tensor([[ 9, 17, 16,  9,  8,  2]], device='cuda:0')
teacher prediction:   tensor([[ 9, 17, 16,  9,  8,  2]], device='cuda:0')
autoregressive:       tensor([[ 1,  9, 17, 16,  9,  8,  2]], device='cuda:0')


## Autoregressive Inference(B>1) Test


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights






# 11. Masks
def create_causal_mask(seq_len, device): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기
    current_device = target_seq.device
    causal_mask = create_causal_mask(seq_len, current_device)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len=5000):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)
        # encoder_output: [B,S,d_model], encoder_attn_weights: list of num_layers, each element: [B,h,S,S]
        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        # decoder_output: [B,T,d_model], decoder_self_attn_weights: list of num_layers, each [B,h,T,T], decoder_cross_attn_weights: list of num_layers, each [B,h,T,S]
        vocab_logits = self.output_linear(decoder_output)
        # vocab_logits: [B,T,tgt_vocab_size]

        return (
                vocab_logits,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

# 13. Copy Task
def generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device): #
    src = torch.randint(3, vocab_size, (batch_size, seq_len), device=device)
    tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
    tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가
    return src, tgt_input, tgt_label # src: [B, S], tgt_input: [B, S+1], tgt_label: [B, S+1]


# 14. Training Loop
vocab_size = 20
pad_idx = 0
bos_idx = 1
eos_idx = 2

seq_len = 5
batch_size = 32

d_model = 32
num_heads = 4
d_ff = 64
num_layers = 2

learning_rate = 1e-3
num_steps = 1000
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = Transformer(vocab_size, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss()


for step in range(num_steps):

    train_src, train_tgt_input, train_tgt_label = generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device)

    optimizer.zero_grad()

    vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(train_src, train_tgt_input)


    logits = vocab_logits.reshape(-1, vocab_logits.size(-1))
    label = train_tgt_label.reshape(-1)

    loss = loss_function(logits, label)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print("Loss: ")
        print(loss)

#16. Autoregressive Inference
@torch.no_grad()
def greedy_decode(model, src, bos_idx, eos_idx, max_new_tokens, device, pad_idx): # src: [B,S]
    model.eval()
    batch_size = src.shape[0]

    # [B, 1] 크기로 bos_idx 채우기
    generated = torch.full((batch_size, 1), bos_idx, dtype=torch.long, device=device)
    # model에 넣어줄 target sequence로 [BOS]만 있는 텐서 넣기

    finished = torch.zeros(batch_size, dtype=torch.bool, device=device) # 텐서 하나를 만들어서 batch의 모든 문장이 EOS로 끝났는지 추적

    # 최대 max_new_tokens개의 새로운 token을 autoregressive하게 생성
    for tokens in range(max_new_tokens):
        vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(src, generated)
        next_token_logits = vocab_logits[:,-1,:] # next_token_logits: [B, vocab_size]
        # vocab_logits에는 모든 timestep의 다음 토큰 예상값이 logits으로 들어있기 때문에
        # 마지막 시점의, 즉 마지막 position의 예상값을 가져오기 위해 T차원을 -1로 가져옴.

        next_token = next_token_logits.argmax(dim=-1) # next_token_logits에서 가장 큰 값의 인덱스는 모델이 예상한 다음 token id. next_token: [B]

        # 이전 step에서 이미 EOS를 생성한 sequence는 이제 PAD만 추가
        next_token = torch.where( # finished 텐서의 값 중 true인 위치에는 torch.full_like(next_token, pad_idx)에서 채워넣고 false면 next_token에서 채워넣기.
        # 즉, 문장이 EOS를 마지막으로 생성해서 끝난상태면 pad token id를 next_token으로 하고, 아니면 그대로 next_token에서 가져옴.
            finished,
            torch.full_like(next_token, pad_idx),
            next_token
        )

        # 이번 step에서 새롭게 EOS를 생성한 sequence까지 finished에 기록
        finished = finished | (next_token == eos_idx)

        next_token = next_token.unsqueeze(-1) # 제일 안쪽 차원을 하나 추가. next_token: [B, 1]
        generated = torch.cat([generated, next_token], dim=1) # generated와 next_token을 붙임. generated: [B, current_length+1]

        if finished.all(): # finished 텐서안의 값이 모두 true면 break
            break

    return generated # generated: [B,T]



# 17. Teacher-forcing vs Autoregressive Inference Test

# 새로운 테스트용 copy batch 생성
test_src, test_tgt_input, test_tgt_label = generate_copy_batch(
    batch_size=4,
    seq_len=seq_len,
    vocab_size=vocab_size,
    bos_idx=bos_idx,
    eos_idx=eos_idx,
    device=device
)

# 1) Teacher-forcing prediction
model.eval()

with torch.no_grad():
    vocab_logits, _, _, _ = model(test_src, test_tgt_input)
    teacher_prediction = vocab_logits.argmax(dim=-1)

# 2) Autoregressive prediction
max_new_tokens = test_src.shape[1] + 1

generated = greedy_decode(
    model,
    test_src,
    bos_idx,
    eos_idx,
    max_new_tokens,
    device,
    pad_idx
)

print("src:                 ", test_src)
print("tgt_input:           ", test_tgt_input)
print("tgt_label:           ", test_tgt_label)
print("teacher prediction:  ", teacher_prediction)
print("autoregressive:      ", generated)

Loss: 
tensor(3.1770, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.9389, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.2774, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.5696, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.3325, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.1091, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0477, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0170, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0153, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0058, device='cuda:0', grad_fn=<NllLossBackward0>)
src:                  tensor([[14,  3,  8, 19, 14],
        [11, 14,  6, 16, 10],
        [16, 19, 11, 13, 16],
        [ 3,  6, 15,  6,  5]], device='cuda:0')
tgt_input:            tensor([[ 1, 14,  3,  8, 19, 14],
        [ 1, 11, 14,  6, 16, 10],
        [ 1, 16, 19, 11, 13, 16],
        [ 1,  3,  6, 15,  6,  5]], device='cud

## Variable Sequence Length Inference Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights






# 11. Masks
def create_causal_mask(seq_len, device): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기
    current_device = target_seq.device
    causal_mask = create_causal_mask(seq_len, current_device)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len=5000):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)
        # encoder_output: [B,S,d_model], encoder_attn_weights: list of num_layers, each element: [B,h,S,S]
        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        # decoder_output: [B,T,d_model], decoder_self_attn_weights: list of num_layers, each [B,h,T,T], decoder_cross_attn_weights: list of num_layers, each [B,h,T,S]
        vocab_logits = self.output_linear(decoder_output)
        # vocab_logits: [B,T,tgt_vocab_size]

        return (
                vocab_logits,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

# 13. Copy Task
def generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device): #
    src = torch.randint(3, vocab_size, (batch_size, seq_len), device=device)
    tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
    tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가
    return src, tgt_input, tgt_label # src: [B, S], tgt_input: [B, S+1], tgt_label: [B, S+1]


# 14. Training Loop
vocab_size = 20
pad_idx = 0
bos_idx = 1
eos_idx = 2

seq_len = 5
batch_size = 32

d_model = 32
num_heads = 4
d_ff = 64
num_layers = 2

learning_rate = 1e-3
num_steps = 1000
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = Transformer(vocab_size, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss(ignore_index=pad_idx)

for step in range(num_steps):

    train_src, train_tgt_input, train_tgt_label = generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device)

    optimizer.zero_grad()

    vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(train_src, train_tgt_input)


    logits = vocab_logits.reshape(-1, vocab_logits.size(-1))
    label = train_tgt_label.reshape(-1)

    loss = loss_function(logits, label)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print("Loss: ")
        print(loss)


#16. Autoregressive Inference
@torch.no_grad()
def greedy_decode(model, src, bos_idx, eos_idx, max_new_tokens, device, pad_idx): # src: [B,S]
    model.eval()
    batch_size = src.shape[0]

    # [B, 1] 크기로 bos_idx 채우기
    generated = torch.full((batch_size, 1), bos_idx, dtype=torch.long, device=device)
    # model에 넣어줄 target sequence로 [BOS]만 있는 텐서 넣기

    finished = torch.zeros(batch_size, dtype=torch.bool, device=device) # 텐서 하나를 만들어서 batch의 각 sequence가 EOS를 생성해 종료되었는지 개별적으로 추적

    # 최대 max_new_tokens개의 새로운 token을 autoregressive하게 생성
    for tokens in range(max_new_tokens):
        vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(src, generated)
        next_token_logits = vocab_logits[:,-1,:] # next_token_logits: [B, vocab_size]
        # vocab_logits에는 모든 timestep의 다음 토큰 예상값이 logits으로 들어있기 때문에
        # 마지막 시점의, 즉 마지막 position의 예상값을 가져오기 위해 T차원을 -1로 가져옴.

        next_token = next_token_logits.argmax(dim=-1) # next_token_logits에서 가장 큰 값의 인덱스는 모델이 예상한 다음 token id. next_token: [B]

        # 이전 step에서 이미 EOS를 생성한 sequence는 이제 PAD만 추가
        next_token = torch.where( # finished 텐서의 값 중 true인 위치에는 torch.full_like(next_token, pad_idx)에서 채워넣고 false면 next_token에서 채워넣기.
        # 즉, 문장이 EOS를 마지막으로 생성해서 끝난상태면 pad token id를 next_token으로 하고, 아니면 그대로 next_token에서 가져옴.
        # finished=True -> pad_idx, False -> 모델이 예측한 next_token 유지
            finished,
            torch.full_like(next_token, pad_idx),
            next_token
        )

        # 이번 step에서 새롭게 EOS를 생성한 sequence까지 finished에 기록
        finished = finished | (next_token == eos_idx)

        next_token = next_token.unsqueeze(-1) # 제일 안쪽 차원을 하나 추가. next_token: [B, 1]
        generated = torch.cat([generated, next_token], dim=1) # generated와 next_token을 붙임. generated: [B, current_length+1]

        if finished.all(): # finished 텐서안의 값이 모두 true면 break
            break

    return generated # generated: [B,T]


# 18. generate_variable_copy_batch
def generate_variable_copy_batch(
    lengths,
    vocab_size,
    bos_idx,
    eos_idx,
    pad_idx,
    device
):

    src_list = []
    tgt_input_list = []
    tgt_label_list = []

    for length in lengths: # lengths는 각 sequence의 길이를 알려주는 1차원 배열

        src = torch.randint(3, vocab_size, (length, ), device=device)

        tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
        tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가

        src_list.append(src)
        tgt_input_list.append(tgt_input)
        tgt_label_list.append(tgt_label)
        # 각 리스트에 추가

    # src_list를 PAD해서 [B, S_max]
    src_batch = pad_sequence(src_list, batch_first=True, padding_value=pad_idx)
    # batch_first=True로 output을 [B, S_max] 형태로 만들고,
    # S_max는 src_list에서 가장 긴 sequence 길이(max(lengths)). pad_sequence가 입력받은 리스트에서 각 텐서의 길이 중 가장 큰 값을 찾아냄.
    # 짧은 sequence의 오른쪽 부족한 부분은 pad_idx로 채움.

    # tgt_input_list를 PAD해서 [B, T_max]
    tgt_input_batch = pad_sequence(tgt_input_list, batch_first=True, padding_value=pad_idx)
    # tgt_input의 각 sequence는 [BOS]가 추가되어 src보다 길이가 1 크므로,
    # padding 후 shape은 [B, T_max], T_max = max(lengths) + 1.

    # tgt_label_list를 PAD해서 [B, T_max]
    tgt_label_batch = pad_sequence(tgt_label_list, batch_first=True, padding_value=pad_idx)
    # tgt_label의 각 sequence는 [EOS]가 추가되어 src보다 길이가 1 크므로,
    # padding 후 shape은 [B, T_max], T_max = max(lengths) + 1.

    return src_batch, tgt_input_batch, tgt_label_batch






# 19. Variable-length Teacher-forcing vs Autoregressive Inference Test

lengths = [2, 5, 3, 4]

test_src, test_tgt_input, test_tgt_label = generate_variable_copy_batch( # test_src: [B,S_max] test_tgt_input,test_tgt_label: [B,T_max]
    lengths=lengths,
    vocab_size=vocab_size,
    bos_idx=bos_idx,
    eos_idx=eos_idx,
    pad_idx=pad_idx,
    device=device
)

# 1) Teacher-forcing prediction
model.eval()

with torch.no_grad():
    vocab_logits, _, _, _ = model(test_src, test_tgt_input)
    teacher_prediction = vocab_logits.argmax(dim=-1)

# 2) Autoregressive prediction
max_new_tokens = max(lengths) + 1

generated = greedy_decode(
    model,
    test_src,
    bos_idx,
    eos_idx,
    max_new_tokens,
    device,
    pad_idx
)

print("src:                 \n", test_src)
print("tgt_input:           \n", test_tgt_input)
print("tgt_label:           \n", test_tgt_label)
print("teacher prediction:  \n", teacher_prediction)
print("autoregressive:      \n", generated)


Loss: 
tensor(3.2296, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.9537, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.1392, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.6049, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.2218, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.1875, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0593, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0191, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0154, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.0093, device='cuda:0', grad_fn=<NllLossBackward0>)
src:                 
 tensor([[10, 11,  0,  0,  0],
        [ 5, 12, 16, 13, 19],
        [13, 13, 13,  0,  0],
        [11,  5,  7,  8,  0]], device='cuda:0')
tgt_input:           
 tensor([[ 1, 10, 11,  0,  0,  0],
        [ 1,  5, 12, 16, 13, 19],
        [ 1, 13, 13, 13,  0,  0],
        [ 1, 11,  5,  7,  8,  0]], device='c

## Variable Sequence Length Training and Inference Test

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence


# 1. Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = d_model
            )

    def forward(self, x):
        x = self.embedding(x)
        x = x * math.sqrt(self.d_model)
        return x

# 2. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # [max_len, d_model] 크기의 0으로 채워진 빈 테이블 생성.
        # 나중에 position을 알려주는 값으로 채우고 embedding된 token이랑 합쳐질 예정.
        pe = torch.zeros(max_len, d_model)
        # 각 row가 몇 번째 position인지 나타냄.
        # sin/cos 계산할거라서 float로 만들기. 0부터 max_len-1까지 만들고 unsqueeze로 제일 안쪽에 차원 추가
        # shape은 [max_len, 1]
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(-1)
        # 각 embedding dimension pair마다 서로 다른 sin/cos 주기를 만들어주는 scale 값들을 생성. shape은 [d_model / 2]  (d_model이 짝수라고 가정)
        # 각 dimension pair라는게 sin/cos을 의미. dim 0과 1 → 같은 frequency 사용, 즉 dim 0 → sin(frequency 0), dim 1 → cos(frequency 0) 이어서 dim 2,3도 같은 frquency 사용..
        # div_term은 사실 각 포지션이 다른 속도의 sin/cos값을 갖게 하기위한 scale임.
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        # position*div_term으로 나온 [max_len, d_model/2] 모양의 텐서가 sin/cos에 들어가서 sin은 짝수, cos은 홀수 column에 들어가게 함. 여기서 column은 d_model. row는 포지션.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # 미리 만든 self.pe는 shape이 [1, max_len, d_model]이라서 그걸 바로
        # x(embedding이 끝난 입력값, shape은 [B, T, d_model])에 더하면 안되서 x.size(1), 즉 토큰개수만큼 잘라줌.
        x = x + self.pe[:, :x.size(1), :]
        return x






# 3. Attention
def attention(Q, K, V, mask=None):
    # Expected shapes: Q/K/V : [B, h, T, d_head], mask: attention score [B,h,T_q,T_k]에 broadcast 가능한 bool tensor
    # ex) Causal mask: [1, 1, T_q, T_k] , Padding mask: [B, 1, 1, T_k]
    d_k = Q.shape[-1]
    K_trans = torch.transpose(K, -2, -1)
    att_scores = Q @ K_trans / math.sqrt(d_k)
    # 만약 mask가 있다면, pytorch의 masked_fill은 mask값이 true인 위치를 바꾸기 때문에,
    # ~mask으로 false, true를 뒤집고 기존에 false였던 부분을 -inf로 채우기(현재 코드에선 true가 허용, false가 차단 인 방식으로 mask를 만들었기 때문)
    # 중요한 부분은, 이 mask가 causal인지 padding인지 알 필요없이 false인 위치를 막음. 그래서 구현이 간단해짐.
    if mask is not None: att_scores = att_scores.masked_fill(~mask, float("-inf"))
    att_weights = F.softmax(att_scores, dim = -1)
    output = att_weights @ V
    return output, att_weights # output: [B, h, T_q, d_v], att_weights: [B, h, T_q, T_k]

# 4. Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None): # Expected query, key, value shape: [B, T, d_model]
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # Q/K/V를 head split 하고 head차원을 앞으로 옮기기
        # ex) Q: [B, T_q, d_model] -> [B, T_q, h, d_head] -> [B, h, T_q, d_head]
        Q = Q.unflatten(-1, (self.num_heads, self.d_head))
        Q = torch.transpose(Q, 1, 2)

        K = K.unflatten(-1, (self.num_heads, self.d_head))
        K = torch.transpose(K, 1, 2)

        V = V.unflatten(-1, (self.num_heads, self.d_head))
        V = torch.transpose(V, 1, 2)

        output, att_weights = attention(Q, K, V, mask)
        # attention에서 나온 output을 다시 원래 shape으로 돌려놓기 head_output [B, h, T_q, d_head] -> [B, T_q, d_model]
        output = torch.transpose(output, 1, 2)
        output = output.flatten(-2) # 숫자하나만 적으면 맨끝에 -1이 default값으로 들어가있어서 (-2,-1) 이렇게 처리됨. 즉 그 두개를 합침.

        output = self.W_O(output)

        return output, att_weights




# 5. Feed-Forwad Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x): # Expected x shape: [B, T, d_model]
        x = self.linear1(x)
        x = self.relu(x)
        output = self.linear2(x)
        return output

# 6. Residual Layer Norm
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.layernorm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        x = x + sublayer_output # Residual connection
        output = self.layernorm(x) # Layernormalization
        return output





# 7. Encoder Block
## 하나의 encoder block이 어떻게 작동해야하는지
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)
    def forward(self, x, src_mask=None):
        attn_output, attn_weights = self.mha(x,x,x, mask=src_mask) # multihead attention에서 새로운 representation과 attetion weight matrix 받기
        residual_ln1 = self.add_norm1(x, attn_output) # 첫번쨰 add+norm 통과
        ffn_output = self.ffn(residual_ln1) # FFN층 통과
        residual_ln2 = self.add_norm2(residual_ln1, ffn_output) # 두번째 add+norm 통과
        return residual_ln2, attn_weights

# 8. Encoder
## 실제로 encoder block들을 조립해서 작동하게 하기
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            EncoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, src_mask=None): # x(encoder input) shape: [B, S, d_model]
        all_attn_weights = []
        for layer in self.layers:
            x, attn_weights = layer(x, src_mask=src_mask)
            all_attn_weights.append(attn_weights)
        return x, all_attn_weights





# 9. Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)

        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.add_norm2 = ResidualLayerNorm(d_model)

        self.ffn = FFN(d_model, d_ff)
        self.add_norm3 = ResidualLayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None ,cross_mask=None): # x(decoder input) shape: [B, T, d_model] , encoder_output shape: [B, S, d_model]
        self_attn_output, self_attn_weights = self.self_attn(x, x, x, mask=self_mask) # q/k/v 모두 decoder input을 가져와서 projection
        residual_ln1 = self.add_norm1(x, self_attn_output)

        cross_attn_output, cross_attn_weights = self.cross_attn(residual_ln1, encoder_output, encoder_output, mask=cross_mask) # cross attention은 k/v를 encoder outputd에서 가져와서 projection
        residual_ln2 = self.add_norm2(residual_ln1, cross_attn_output)

        ffn_output = self.ffn(residual_ln2)
        residual_ln3 = self.add_norm3(residual_ln2, ffn_output)
        # self_attn_weights: [B,h, T, T] , cross_atnn_weights: [B, H, T, S]
        return residual_ln3, self_attn_weights, cross_attn_weights

# 10. Decoder
## 실제로 decoder block들을 조립해서 작동하게 하기
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            DecoderBlock(d_model, num_heads, d_ff) for i in range(num_layers)
            )

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None): # x(decoder input) shape: [B, T, d_model]
        all_self_attn_weights =[]
        all_cross_attn_weights =[]
        for layer in self.layers:
            x, self_attn_weights, cross_attn_weights = layer(x, encoder_output, self_mask=self_mask, cross_mask=cross_mask)
            all_self_attn_weights.append(self_attn_weights)
            all_cross_attn_weights.append(cross_attn_weights)
        return x, all_self_attn_weights, all_cross_attn_weights






# 11. Masks
def create_causal_mask(seq_len, device): # Decoder self-attention에서 미래 target token을 attend하지 못하게 함.
    # seq_len = target sequence length T.

    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Decoder self-attention에서는 T_q = T_k = T이므로 [T_q, T_k] = [T, T] lower-triangular mask를 만듦.

    mask = mask.unsqueeze(dim=0)
    mask = mask.unsqueeze(dim=0).bool()
    # mask: [1, 1, T, T]
    # 앞 쪽에 두 차원을 추가해서 나중에 [B, h, T_q, T_k] shape을 가진 attention score와 계산될 예정.
    return mask

def create_padding_mask(seq, pad_idx): # 현재 K/V를 제공하는 sequence의 PAD 위치를 아무 Query도 참고하지 못하게 함.

    # seq 텐서와 PAD token의 정수 id를 받음. seq: [B, T_k]
    mask = seq != pad_idx
    mask = mask.unsqueeze(1).unsqueeze(2) # mask: [B, 1, 1, T_k] -> # head dimension과 query dimension에 broadcasting되도록 함.
    # 왜냐면 모든 head의 모든 query에서 PAD key를 참고하지 못하게 해야해서.

    return mask # 따라서 각 batch에서 모든 head와 모든 query가 PAD key를 보지 못함.

def create_decoder_mask(target_seq, pad_idx):# target_seq: [B, T]
    # Decoder self-attention용 causal mask + target padding mask 생성.
    seq_len = target_seq.shape[1]
    # target_seq: [B, T] 에서 T 가져오기
    current_device = target_seq.device
    causal_mask = create_causal_mask(seq_len, current_device)
    # causal mask: [1, 1, T_q, T_k] = [1, 1, T, T]

    padding_mask = create_padding_mask(target_seq, pad_idx)
    # paddin mask: [B, 1, 1, T_k]

    combined_mask = causal_mask & padding_mask # combined mask: [B, 1, T, T_k]
    return combined_mask


# 12. Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx, max_len=5000):
        super().__init__()
        self.src_token_embedding = TokenEmbedding(src_vocab_size, d_model)
        self.tgt_token_embedding = TokenEmbedding(tgt_vocab_size, d_model)

        self.src_positional_encoding = PositionalEncoding(d_model, max_len)
        self.tgt_positional_encoding = PositionalEncoding(d_model, max_len)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.pad_idx = pad_idx

        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt): # source sequence: [B, S] , target sequence: [B, T]
        src_x = self.src_token_embedding(src)
        src_x = self.src_positional_encoding(src_x)

        src_mask = create_padding_mask(seq=src, pad_idx=self.pad_idx) # src_mask는 padding mask만
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.
        encoder_output, encoder_attn_weights = self.encoder(src_x, src_mask)
        # encoder_output: [B,S,d_model], encoder_attn_weights: list of num_layers, each element: [B,h,S,S]
        tgt_x = self.tgt_token_embedding(tgt)
        tgt_x = self.tgt_positional_encoding(tgt_x)

        tgt_mask = create_decoder_mask(target_seq=tgt, pad_idx=self.pad_idx) # tgt_mask는 causal mask+padding mask
        # padding mask를 만들때는 embedding하기전의 src sequence를 가져와야지 token id를 보고 pad인지 확인할 수 있음.

        decoder_output, decoder_self_attn_weights, decoder_cross_attn_weights = self.decoder(tgt_x, encoder_output, tgt_mask, src_mask)
        # decoder_output: [B,T,d_model], decoder_self_attn_weights: list of num_layers, each [B,h,T,T], decoder_cross_attn_weights: list of num_layers, each [B,h,T,S]
        vocab_logits = self.output_linear(decoder_output)
        # vocab_logits: [B,T,tgt_vocab_size]

        return (
                vocab_logits,
                encoder_attn_weights,
                decoder_self_attn_weights,
                decoder_cross_attn_weights
                )

# 13. Copy Task
def generate_copy_batch(batch_size, seq_len, vocab_size, bos_idx, eos_idx, device): #
    src = torch.randint(3, vocab_size, (batch_size, seq_len), device=device)
    tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
    tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가
    return src, tgt_input, tgt_label # src: [B, S], tgt_input: [B, S+1], tgt_label: [B, S+1]


# 18. generate_variable_copy_batch
def generate_variable_copy_batch(
    lengths,
    vocab_size,
    bos_idx,
    eos_idx,
    pad_idx,
    device
):

    src_list = []
    tgt_input_list = []
    tgt_label_list = []

    for length in lengths: # lengths는 각 sequence의 길이를 알려주는 1차원 배열

        src = torch.randint(3, vocab_size, (length, ), device=device)

        tgt_input = F.pad(src, (1,0), value=bos_idx) # (1,0)을 하면 앞에 하나 추가
        tgt_label = F.pad(src, (0,1), value=eos_idx) # (0,1)을 하면 뒤에 하나 추가

        src_list.append(src)
        tgt_input_list.append(tgt_input)
        tgt_label_list.append(tgt_label)
        # 각 리스트에 추가

    # src_list를 PAD해서 [B, S_max]
    src_batch = pad_sequence(src_list, batch_first=True, padding_value=pad_idx)
    # batch_first=True로 output을 [B, S_max] 형태로 만들고,
    # S_max는 src_list에서 가장 긴 sequence 길이(max(lengths)).
    # 짧은 sequence의 오른쪽 부족한 부분은 pad_idx로 채움.

    # tgt_input_list를 PAD해서 [B, T_max]
    tgt_input_batch = pad_sequence(tgt_input_list, batch_first=True, padding_value=pad_idx)
    # tgt_input의 각 sequence는 [BOS]가 추가되어 src보다 길이가 1 크므로,
    # padding 후 shape은 [B, T_max], T_max = max(lengths) + 1.

    # tgt_label_list를 PAD해서 [B, T_max]
    tgt_label_batch = pad_sequence(tgt_label_list, batch_first=True, padding_value=pad_idx)
    # tgt_label의 각 sequence는 [EOS]가 추가되어 src보다 길이가 1 크므로,
    # padding 후 shape은 [B, T_max], T_max = max(lengths) + 1.

    return src_batch, tgt_input_batch, tgt_label_batch

# 14. Training Loop
vocab_size = 20
pad_idx = 0
bos_idx = 1
eos_idx = 2

lengths = [2, 5, 3, 4, 8, 6, 9, 4, 5, 3, 2, 5, 3, 4, 8, 6, 9, 4, 5, 3, 2, 5, 3, 4, 8, 6, 9, 4, 5, 3, 5, 3, 7]

d_model = 32
num_heads = 4
d_ff = 64
num_layers = 2

learning_rate = 1e-3
num_steps = 1000
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = Transformer(vocab_size, vocab_size, d_model, num_heads, d_ff, num_layers, pad_idx).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss(ignore_index=pad_idx)

for step in range(num_steps):

    train_src, train_tgt_input, train_tgt_label = generate_variable_copy_batch( # train_src: [B,S_max] train_tgt_input,train_tgt_label: [B,T_max]
    lengths=lengths,
    vocab_size=vocab_size,
    bos_idx=bos_idx,
    eos_idx=eos_idx,
    pad_idx=pad_idx,
    device=device
)
    optimizer.zero_grad()

    vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(train_src, train_tgt_input)


    logits = vocab_logits.reshape(-1, vocab_logits.size(-1))
    label = train_tgt_label.reshape(-1)

    loss = loss_function(logits, label)

    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print("Loss: ")
        print(loss)

#16. Autoregressive Inference
@torch.no_grad()
def greedy_decode(model, src, bos_idx, eos_idx, max_new_tokens, device, pad_idx): # src: [B,S]
    model.eval()
    batch_size = src.shape[0]

    # [B, 1] 크기로 bos_idx 채우기
    generated = torch.full((batch_size, 1), bos_idx, dtype=torch.long, device=device)
    # model에 넣어줄 target sequence로 [BOS]만 있는 텐서 넣기

    finished = torch.zeros(batch_size, dtype=torch.bool, device=device) # 텐서 하나를 만들어서 batch의 각 sequence가 EOS를 생성해 종료되었는지 개별적으로 추적

    # 최대 max_new_tokens개의 새로운 token을 autoregressive하게 생성
    for tokens in range(max_new_tokens):
        vocab_logits,encoder_attn_weights,decoder_self_attn_weights,decoder_cross_attn_weights = model(src, generated)
        next_token_logits = vocab_logits[:,-1,:] # next_token_logits: [B, vocab_size]
        # vocab_logits에는 모든 timestep의 다음 토큰 예상값이 logits으로 들어있기 때문에
        # 마지막 시점의, 즉 마지막 position의 예상값을 가져오기 위해 T차원을 -1로 가져옴.

        next_token = next_token_logits.argmax(dim=-1) # next_token_logits에서 가장 큰 값의 인덱스는 모델이 예상한 다음 token id. next_token: [B]

        # 이전 step에서 이미 EOS를 생성한 sequence는 이제 PAD만 추가
        next_token = torch.where( # finished 텐서의 값 중 true인 위치에는 torch.full_like(next_token, pad_idx)에서 채워넣고 false면 next_token에서 채워넣기.
        # 즉, 문장이 EOS를 마지막으로 생성해서 끝난상태면 pad token id를 next_token으로 하고, 아니면 그대로 next_token에서 가져옴.
        # finished=True -> pad_idx, False -> 모델이 예측한 next_token 유지
            finished,
            torch.full_like(next_token, pad_idx),
            next_token
        )

        # 이번 step에서 새롭게 EOS를 생성한 sequence까지 finished에 기록
        finished = finished | (next_token == eos_idx)

        next_token = next_token.unsqueeze(-1) # 제일 안쪽 차원을 하나 추가. next_token: [B, 1]
        generated = torch.cat([generated, next_token], dim=1) # generated와 next_token을 붙임. generated: [B, current_length+1]

        if finished.all(): # finished 텐서안의 값이 모두 true면 break
            break

    return generated # generated: [B,T]






# 19. Variable-length Teacher-forcing vs Autoregressive Inference Test

lengths = [2, 5, 3, 4]

test_src, test_tgt_input, test_tgt_label = generate_variable_copy_batch( # test_src: [B,S_max] test_tgt_input,test_tgt_label: [B,T_max]
    lengths=lengths,
    vocab_size=vocab_size,
    bos_idx=bos_idx,
    eos_idx=eos_idx,
    pad_idx=pad_idx,
    device=device
)

# 1) Teacher-forcing prediction
model.eval()

with torch.no_grad():
    vocab_logits, _, _, _ = model(test_src, test_tgt_input)
    teacher_prediction = vocab_logits.argmax(dim=-1)

# 2) Autoregressive prediction
max_new_tokens = max(lengths) + 1

generated = greedy_decode(
    model,
    test_src,
    bos_idx,
    eos_idx,
    max_new_tokens,
    device,
    pad_idx
)

print("src:                 \n", test_src)
print("tgt_input:           \n", test_tgt_input)
print("tgt_label:           \n", test_tgt_label)
print("teacher prediction:  \n", teacher_prediction)
print("autoregressive:      \n", generated)


Loss: 
tensor(3.1579, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.9900, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.5131, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(1.1844, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.8584, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.6418, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.5174, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.4454, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.3656, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss: 
tensor(0.3240, device='cuda:0', grad_fn=<NllLossBackward0>)
src:                 
 tensor([[10,  6,  0,  0,  0],
        [ 9, 18, 14,  5, 17],
        [14,  7, 15,  0,  0],
        [ 5, 10, 19, 16,  0]], device='cuda:0')
tgt_input:           
 tensor([[ 1, 10,  6,  0,  0,  0],
        [ 1,  9, 18, 14,  5, 17],
        [ 1, 14,  7, 15,  0,  0],
        [ 1,  5, 10, 19, 16,  0]], device='c